In [1]:
#Modify the path to a directory on your machine
import os
os.environ["CRDS_PATH"] = "/home/hailin/Documents/CRDS"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# Packages that allow us to get information about objects:
import asdf
import copy
import shutil

# Numpy library:
import numpy as np

# For downloading data
import requests

# Astropy tools:
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch

import matplotlib.pyplot as plt
import matplotlib as mpl

# Plotting tools:
from pipeline1_plotting_tools import download_files, plot_jump, plot_jumps, plot_ramp, plot_ramps, show_image, side_by_side

# Use this version for non-interactive plots (easier scrolling of the notebook)
%matplotlib inline

# Use this version (outside of Jupyter Lab) if you want interactive plots
#%matplotlib notebook

# List of possible data quality flags
from jwst.datamodels import dqflags

# The entire calwebb_detector1 pipeline
from jwst.pipeline import calwebb_detector1

# Individual steps that make up calwebb_detector1
from jwst.dq_init import DQInitStep
from jwst.saturation import SaturationStep
from jwst.superbias import SuperBiasStep
from jwst.ipc import IPCStep                                                                                    
from jwst.refpix import RefPixStep                                                                
from jwst.linearity import LinearityStep
from jwst.persistence import PersistenceStep
from jwst.dark_current import DarkCurrentStep
from jwst.jump import JumpStep
from jwst.ramp_fitting import RampFitStep
from jwst import datamodels

import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from pathlib import Path

import jwst
print(jwst.__version__)

1.15.1


In [4]:
data_path = Path('../data/JWST/3000s_exposure_blind')
data_behav = np.zeros((20,4))
i=0
for item in data_path.iterdir():
    input_file_base = item.name
    jump_file = '../data/JWST/3000s_exposure_blind/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    n_group = jump.data.shape[1]
    photo_start = 0
    photo_end   = n_group - 1
    # Create a 4-dimensional map of ANY flags
    dq_map_2d = np.sum(jump.groupdq[0, :, :, :] > 0, axis=0)
    # Determine how many pixels are not GOOD
    dq_map_indexes = np.where(dq_map_2d > 0)
    bad_pix = np.sum(dq_map_2d > 0)
    total_pix = 2048 * 2048
    saturated = (jump.groupdq & dqflags.pixel['SATURATED'] > 0)
    saturated_2d = np.sum(saturated[0, :, :, :], axis=0)
    saturated_ratio = len(np.where((saturated_2d > 0))[0]) / total_pix
    persistence = (jump.groupdq & dqflags.pixel['PERSISTENCE'] > 0)
    persistence_2d = np.sum(persistence[0, :, :, :], axis=0)
    persistence_ratio = len(np.where((persistence_2d > 0))[0]) / total_pix
    jump_det = (jump.groupdq & dqflags.pixel['JUMP_DET'] > 0)
    jump_det_2d = np.sum(jump_det[0, :, :, :], axis=0)
    jump_det_ratio = len(np.where((jump_det_2d > 0))[0]) / total_pix
    bad_ratio = bad_pix / total_pix
    data_behav[i,0] = saturated_ratio
    data_behav[i,1] = persistence_ratio
    data_behav[i,2] = jump_det_ratio
    data_behav[i,3] = bad_ratio
    i += 1

In [ ]:
data_behav[:,0] + data_behav[:,2]

In [ ]:
data_behav

In [ ]:
np.average(data_behav[:,0])

In [ ]:
np.average(data_behav[:,2])

In [ ]:
np.average(data_behav[:,3])

In [ ]:
np.std(data_behav[:,3], ddof=1)

In [ ]:
data_path = Path('../data/JWST/3000s_exposure_blind')
for item in data_path.iterdir():
    input_file_base = item.name
    jump_file = '../data/JWST/3000s_exposure_blind/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    n_group = jump.data.shape[1]
    photo_start = 0
    photo_end   = n_group - 1
    # bin 的范围
    dn_min = -200
    dn_max = 400
    # Create a 4-dimensional map of ANY flags
    dq_map_2d = np.sum(jump.groupdq[0, :, :, :] > 0, axis=0)
    # Determine how many pixels are not GOOD
    dq_map_indexes = np.where(dq_map_2d > 0)
    bad_pix = np.sum(dq_map_2d > 0)
    total_pix = 2048 * 2048
    pre_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
    photo_zero_bad = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
    photo_zero_bad[dq_map_indexes] = 0

    # c stands for control:
    c_delta_mask   = True
    c_bad_mosaic   = True
    c_secondary    = True
    c_edge         = True
    c_brightness   = True

    mosaic_size = 16
    mask_perc_thr_1  = 0.9
    secondary_size_1 = 3
    mask_perc_thr_2  = 0.6
    secondary_size_2 = 1
    edge_width = 32
    brightness_thr  = 150
    brightness_size = 2
    delta_thr = 60

    mask = np.zeros_like(pre_mask, dtype=bool)

    # 这次我们直接从超亮pixel的分布入手，mask掉超亮pixel周围的pixel
    # 并且这次设计两套bad mask。越亮的区域需要的secondary越大。
    bad_mask = np.zeros_like(pre_mask, dtype=bool)
    mosaic_area = mosaic_size**2
    mosaic_half = int(mosaic_size / 2)
    N_mosaic = int(2048 / mosaic_size)
    bad_mosaic_1 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    bad_mosaic_2 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    if c_bad_mosaic:
        bad_mosaic_count = np.zeros((N_mosaic, N_mosaic))
        # 现在可以直接从dq_map_indexes出发，对其进行pixel内的计数
        bad_pix_map = (dq_map_2d > 0)
        # 热力图，但是不带已经flagged的
        for y in range(N_mosaic):  # jwst 指标先y后x
            for x in range(N_mosaic):
                bad_mosaic_count[y,x] = np.sum(bad_pix_map[y*mosaic_size:(y+1)*mosaic_size, x*mosaic_size:(x+1)*mosaic_size])
        # 把马赛克化的mask汇总于此：
        bad_mosaic_1  = (bad_mosaic_count/mosaic_area >mask_perc_thr_1)
        bad_mosaic_2  = (bad_mosaic_count/mosaic_area >mask_perc_thr_2)

    #cols_mosaic = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    #cols_mosaic[:,-14:-12] = True
    #cols_mosaic[:,28:30] = True
    #cols_mosaic[:,-4:-2] = True

    secondary_mosaic_1 = np.zeros((N_mosaic, N_mosaic), dtype=bool) #coule be the most important
    secondary_mosaic_2 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    if c_secondary:
        for y in range(N_mosaic):  # jwst 指标先y后x
            for x in range(N_mosaic):
                if  bad_mosaic_1[y,x] :
                    y_min = np.max([y-secondary_size_1,0])
                    x_min = np.max([x-secondary_size_1,0])
                    y_max = np.min([y+secondary_size_1,N_mosaic-1])
                    x_max = np.min([x+secondary_size_1,N_mosaic-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= secondary_size_1**2:
                                secondary_mosaic_1[y2,x2] = True
                if  bad_mosaic_2[y,x] :
                    y_min = np.max([y-secondary_size_2,0])
                    x_min = np.max([x-secondary_size_2,0])
                    y_max = np.min([y+secondary_size_2,N_mosaic-1])
                    x_max = np.min([x+secondary_size_2,N_mosaic-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= secondary_size_2**2:
                                secondary_mosaic_2[y2,x2] = True

    mask_mosaic = bad_mosaic_1  + secondary_mosaic_1 + bad_mosaic_2  + secondary_mosaic_2

    for y in range(N_mosaic):  # jwst 指标先y后x
        for x in range(N_mosaic):
            if  mask_mosaic[y,x]:
                bad_mask[y*mosaic_size:(y+1)*mosaic_size, x*mosaic_size:(x+1)*mosaic_size] = True

    edge_mask = np.zeros_like(photo, dtype=bool)
    if c_edge:
        edge_mask[0:edge_width,:] = True
        edge_mask[-edge_width:,:] = True
        edge_mask[:,0:edge_width] = True
        edge_mask[:,-edge_width:] = True

    # brightness_mask排除剩余的过亮点以及周围
    brightness_mask = np.zeros_like(photo, dtype=bool)

    if c_brightness:
        for y in range(2048):
            for x in range(2048):
                if  np.abs(pre_mask[y,x]) > brightness_thr: # not mask[y,x] and not dq_map_2d[y,x]
                    y_min = np.max([y-brightness_size,0])
                    x_min = np.max([x-brightness_size,0])
                    y_max = np.min([y+brightness_size,2048-1])
                    x_max = np.min([x+brightness_size,2048-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= brightness_size**2:
                                brightness_mask[y2,x2] = True


    # dev：外加一个delta mask，相当于手搓版的jump mask
    delta_mask = np.zeros_like(photo, dtype=bool)
    if c_delta_mask:
        jump_data = jump.data[0]
        delta = (jump_data - np.roll(jump_data, 1, axis=0))[1:,:,:]
        delta_map_2d = np.sum(np.abs(delta[:,:,:]) > delta_thr, axis=0)
        #delta_map_2d = np.sum(delta[:,:,:] < -delta_thr, axis=0)
        delta_mask   = (delta_map_2d > 0)

    mask = bad_mask + edge_mask + brightness_mask + delta_mask

    pixel_after_pipeline = total_pix - bad_pix
    pipeline_bad = dq_map_2d + bad_mask
    pixel_after_pipeline_bad = total_pix - np.sum(pipeline_bad>0)
    bad_ratio = 1 - pixel_after_pipeline_bad/pixel_after_pipeline

    pipeline_bad_edge = dq_map_2d + bad_mask + edge_mask
    pixel_after_pipeline_bad_edge = total_pix - np.sum(pipeline_bad_edge>0)
    edge_ratio = 1 - pixel_after_pipeline_bad_edge/pixel_after_pipeline_bad

    pip_bad_edge_bright = dq_map_2d + bad_mask + edge_mask + brightness_mask
    pixel_after_pip_bad_edge_bright = total_pix - np.sum(pip_bad_edge_bright>0)
    bright_ratio = 1 - pixel_after_pip_bad_edge_bright / pixel_after_pipeline_bad_edge

    full = dq_map_2d + bad_mask + edge_mask + brightness_mask + delta_mask
    pixel_after_full = total_pix - np.sum(full>0)
    delta_ratio = 1 - pixel_after_full / pixel_after_pip_bad_edge_bright

    # 现在mask和原bad pixel mask合并
    masked_indexes = np.where(mask)
    raw_photo_3 = pre_mask
    raw_photo_3[dq_map_indexes] = np.nan
    raw_photo_3[masked_indexes] = np.nan
    raw_photo_flatten_3 = raw_photo_3.flatten()
    nan_mask_3 = np.isnan(raw_photo_flatten_3)
    photo_masked = raw_photo_flatten_3[~nan_mask_3]
    #masked_counts, masked_bin_edges = np.histogram(photo_masked, bins=range(dn_min, dn_max+1))
    raw_photo_2 = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
    raw_photo_2[dq_map_indexes] = np.nan
    raw_photo_flatten_2 = raw_photo_2.flatten()
    nan_mask_2 = np.isnan(raw_photo_flatten_2)
    photo_good_flatten = raw_photo_flatten_2[~nan_mask_2]
    plt.figure(figsize=(12, 5))
    good_counts, good_bin_edges, good_patched = plt.hist(photo_good_flatten, bins=range(dn_min, dn_max+1), alpha=0.8, color='blue',edgecolor='black',log=True)
    masked_counts, masked_bin_edges, masked_patched = plt.hist(photo_masked, bins=range(dn_min, dn_max+1), alpha=0.8, color='red',edgecolor='black',log=True)
    # 设置边缘颜色的透明度
    for patch in plt.gca().patches:
        patch.set_edgecolor((1, 1, 1, 0.2))  # RGBA格式 (红, 绿, 蓝, 透明度)
    plt.title('Pixel Value Distribution')
    plt.xlabel('DN')
    plt.ylabel('Number of Pixels')
    plt.ylim(1,1e6)
    legend_elements = [
        Line2D([0], [0], color='b', lw=2, label='only JWST pipeline flags'),
        Line2D([0], [0], color='r', lw=2, label='w/ halo mask')
    ]
    plt.legend(handles=legend_elements, fontsize=12, loc='upper right', title_fontsize='13', frameon=True)
    good_peak   = np.argmax(good_counts) + dn_min
    masked_peak = np.argmax(masked_counts) + dn_min
    plt.savefig('./results/blind/' + input_file_base + '_histogram_after_mask.jpg')
    plt.close()

    # 保存结果
    result = np.array([range(dn_min,dn_max), masked_counts]).T
    np.savetxt('./results/blind/'+ input_file_base +'.txt',result, header = '{}'.format(bad_pix / total_pix))
    # 还有什么别的要保存的？
    norm = ImageNormalize(photo_zero_bad, interval=ManualInterval(vmin=20, vmax=200),stretch=LogStretch())
    fig = plt.figure(figsize=(60, 60))
    ax = fig.add_subplot(1, 1, 1)
    im = ax.imshow(photo_zero_bad, origin='lower',norm = norm)
    fig.colorbar(im,label = 'DN')
    plt.xlabel('Pixel column')
    plt.ylabel('Pixel row')
    plt.savefig('./results/blind/' + input_file_base + '_zero_bad.jpg')
    plt.close()

    print(('{} pixels ({:.2f}% of the detector) has been flagged'.format(bad_pix, 100. * bad_pix / total_pix)))
    print('{}% pixels left after mask'.format(np.sum(masked_counts)/np.sum(good_counts)*100))
    print('peak of flagged but unmasked image at {}'.format(good_peak))
    print('peak of flagged and masked image at {}'.format(masked_peak))
    print(input_file_base + ' analysis completed')

In [2]:
data_path = Path('../data/JWST/3000s_exposure_blind')
data_behav_mask = np.zeros((20,6))
i=0
for item in data_path.iterdir():
    input_file_base = item.name
    jump_file = '../data/JWST/3000s_exposure_blind/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    n_group = jump.data.shape[1]
    photo_start = 0
    photo_end   = n_group - 1
    # bin 的范围
    dn_min = -200
    dn_max = 400
    # Create a 4-dimensional map of ANY flags
    dq_map_2d = np.sum(jump.groupdq[0, :, :, :] > 0, axis=0)
    # Determine how many pixels are not GOOD
    dq_map_indexes = np.where(dq_map_2d > 0)
    bad_pix = np.sum(dq_map_2d > 0)
    total_pix = 2048 * 2048
    pre_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
    photo_zero_bad = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
    photo_zero_bad[dq_map_indexes] = 0

    # c stands for control:
    c_delta_mask   = True
    c_bad_mosaic   = True
    c_secondary    = True
    c_edge         = True
    c_brightness   = True

    mosaic_size = 16
    mask_perc_thr_1  = 0.9
    secondary_size_1 = 3
    mask_perc_thr_2  = 0.6
    secondary_size_2 = 1
    edge_width = 32
    brightness_thr  = 150
    brightness_size = 2
    delta_thr = 60

    mask = np.zeros_like(pre_mask, dtype=bool)

    # 这次我们直接从超亮pixel的分布入手，mask掉超亮pixel周围的pixel
    # 并且这次设计两套bad mask。越亮的区域需要的secondary越大。
    bad_mask = np.zeros_like(pre_mask, dtype=bool)
    mosaic_area = mosaic_size**2
    mosaic_half = int(mosaic_size / 2)
    N_mosaic = int(2048 / mosaic_size)
    bad_mosaic_1 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    bad_mosaic_2 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    if c_bad_mosaic:
        bad_mosaic_count = np.zeros((N_mosaic, N_mosaic))
        # 现在可以直接从dq_map_indexes出发，对其进行pixel内的计数
        bad_pix_map = (dq_map_2d > 0)
        # 热力图，但是不带已经flagged的
        for y in range(N_mosaic):  # jwst 指标先y后x
            for x in range(N_mosaic):
                bad_mosaic_count[y,x] = np.sum(bad_pix_map[y*mosaic_size:(y+1)*mosaic_size, x*mosaic_size:(x+1)*mosaic_size])
        # 把马赛克化的mask汇总于此：
        bad_mosaic_1  = (bad_mosaic_count/mosaic_area >mask_perc_thr_1)
        bad_mosaic_2  = (bad_mosaic_count/mosaic_area >mask_perc_thr_2)

    #cols_mosaic = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    #cols_mosaic[:,-14:-12] = True
    #cols_mosaic[:,28:30] = True
    #cols_mosaic[:,-4:-2] = True

    secondary_mosaic_1 = np.zeros((N_mosaic, N_mosaic), dtype=bool) #coule be the most important
    secondary_mosaic_2 = np.zeros((N_mosaic, N_mosaic), dtype=bool)
    if c_secondary:
        for y in range(N_mosaic):  # jwst 指标先y后x
            for x in range(N_mosaic):
                if  bad_mosaic_1[y,x] :
                    y_min = np.max([y-secondary_size_1,0])
                    x_min = np.max([x-secondary_size_1,0])
                    y_max = np.min([y+secondary_size_1,N_mosaic-1])
                    x_max = np.min([x+secondary_size_1,N_mosaic-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= secondary_size_1**2:
                                secondary_mosaic_1[y2,x2] = True
                if  bad_mosaic_2[y,x] :
                    y_min = np.max([y-secondary_size_2,0])
                    x_min = np.max([x-secondary_size_2,0])
                    y_max = np.min([y+secondary_size_2,N_mosaic-1])
                    x_max = np.min([x+secondary_size_2,N_mosaic-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= secondary_size_2**2:
                                secondary_mosaic_2[y2,x2] = True

    mask_mosaic = bad_mosaic_1  + secondary_mosaic_1 + bad_mosaic_2  + secondary_mosaic_2

    for y in range(N_mosaic):  # jwst 指标先y后x
        for x in range(N_mosaic):
            if  mask_mosaic[y,x]:
                bad_mask[y*mosaic_size:(y+1)*mosaic_size, x*mosaic_size:(x+1)*mosaic_size] = True

    edge_mask = np.zeros_like(pre_mask, dtype=bool)
    if c_edge:
        edge_mask[0:edge_width,:] = True
        edge_mask[-edge_width:,:] = True
        edge_mask[:,0:edge_width] = True
        edge_mask[:,-edge_width:] = True

    # brightness_mask排除剩余的过亮点以及周围
    brightness_mask = np.zeros_like(pre_mask, dtype=bool)

    if c_brightness:
        for y in range(2048):
            for x in range(2048):
                if  np.abs(pre_mask[y,x]) > brightness_thr: # not mask[y,x] and not dq_map_2d[y,x]
                    y_min = np.max([y-brightness_size,0])
                    x_min = np.max([x-brightness_size,0])
                    y_max = np.min([y+brightness_size,2048-1])
                    x_max = np.min([x+brightness_size,2048-1])
                    # circle mask:
                    for y2 in range(y_min,y_max+1):
                        for x2 in range(x_min,x_max+1):
                            if (y2-y)**2 + (x2-x)**2 <= brightness_size**2:
                                brightness_mask[y2,x2] = True


    # dev：外加一个delta mask，相当于手搓版的jump mask
    delta_mask = np.zeros_like(pre_mask, dtype=bool)
    if c_delta_mask:
        jump_data = jump.data[0]
        delta = (jump_data - np.roll(jump_data, 1, axis=0))[1:,:,:]
        delta_map_2d = np.sum(np.abs(delta[:,:,:]) > delta_thr, axis=0)
        #delta_map_2d = np.sum(delta[:,:,:] < -delta_thr, axis=0)
        delta_mask   = (delta_map_2d > 0)

    mask = bad_mask + edge_mask + brightness_mask + delta_mask

    pixel_after_pipeline = total_pix - bad_pix
    pipeline_bad = dq_map_2d + bad_mask
    pixel_after_pipeline_bad = total_pix - np.sum(pipeline_bad>0)
    bad_ratio = 1 - pixel_after_pipeline_bad/pixel_after_pipeline

    pipeline_bad_edge = dq_map_2d + bad_mask + edge_mask
    pixel_after_pipeline_bad_edge = total_pix - np.sum(pipeline_bad_edge>0)
    edge_ratio = 1 - pixel_after_pipeline_bad_edge/pixel_after_pipeline_bad

    pip_bad_edge_bright = dq_map_2d + bad_mask + edge_mask + brightness_mask
    pixel_after_pip_bad_edge_bright = total_pix - np.sum(pip_bad_edge_bright>0)
    bright_ratio = 1 - pixel_after_pip_bad_edge_bright / pixel_after_pipeline_bad_edge

    full = dq_map_2d + bad_mask + edge_mask + brightness_mask + delta_mask
    pixel_after_full = total_pix - np.sum(full>0)
    delta_ratio = 1 - pixel_after_full / pixel_after_pip_bad_edge_bright
    
    data_behav_mask[i,0] = bad_pix/2048**2
    data_behav_mask[i,1] = bad_ratio
    data_behav_mask[i,2] = edge_ratio
    data_behav_mask[i,3] = bright_ratio
    data_behav_mask[i,4] = delta_ratio
    data_behav_mask[i,5] = pixel_after_full / 2048**2
    
    print(i)
    i += 1

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


In [22]:
testdata = (1-data_behav_mask[:,0]) * (1-data_behav_mask[:,1]) * (1-data_behav_mask[:,2]) * (1-data_behav_mask[:,3]) * (1-data_behav_mask[:,4])
#testdata = data_behav_mask[:,5]
print(np.average(testdata))
print(np.std(testdata, ddof=1))

0.04442607164382935
0.008556933375150764


In [11]:
testdata

array([0.08884852, 0.12491376, 0.11726315, 0.12606722, 0.11221491,
       0.14555858, 0.09261869, 0.1447413 , 0.13200397, 0.10193486,
       0.09675435, 0.11857831, 0.12051983, 0.11351953, 0.11692922,
       0.14312365, 0.11508114, 0.1135577 , 0.12730787, 0.13972732])

In [3]:
np.savetxt('./results/data_behav_mask.txt',data_behav_mask)

In [ ]:
# unmaksed, 过完pipeline屏蔽掉flagged pixels之后直出
data_path = Path('../data/JWST/3000s_exposure')
for item in data_path.iterdir():
    input_file_base = item.name
    jump_file = '../data/JWST/3000s_exposure/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    n_group = jump.data.shape[1]
    photo_start = 0
    photo_end   = n_group - 1
    # bin 的范围
    dn_min = -200
    dn_max = 400
    # Create a 4-dimensional map of ANY flags
    dq_map_2d = np.sum(jump.groupdq[0, :, :, :] > 0, axis=0)
    # Determine how many pixels are not GOOD
    dq_map_indexes = np.where(dq_map_2d > 0)
    bad_pix = np.sum(dq_map_2d > 0)
    total_pix = 2048 * 2048
    pre_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]

    raw_photo_2 = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]
    raw_photo_2[dq_map_indexes] = np.nan
    raw_photo_flatten_2 = raw_photo_2.flatten()
    nan_mask_2 = np.isnan(raw_photo_flatten_2)
    photo_good_flatten = raw_photo_flatten_2[~nan_mask_2]
    good_counts, good_bin_edges = np.histogram(photo_good_flatten, bins=range(dn_min, dn_max+1))

    # 保存结果
    result = np.array([range(dn_min,dn_max), good_counts]).T
    np.savetxt('./results/unmasked/'+ input_file_base +'_unmasked.txt',result)

    print(input_file_base + ' analysis completed')